# 01 · PCA Preprocessing for XGB + FT-Transformer (Home Credit)

Tests whether PCA-whitening the (already preprocessed) numerical feature space helps either:

- **XGBoost**, where rotated/whitened features may interact better with   axis-aligned splits;
- **FT-Transformer**, where decorrelating the inputs may improve   attention quality.

Both raw-feature and PCA-feature baselines are trained side-by-side for direct comparison.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/preprocessed_data_home_credit.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/home_credit")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Home Credit data + drop the leakage / low-signal columns
# identified during initial EDA.
df = pd.read_parquet(DATA_PATH)

DROP_COLUMNS_HC = [
    "FLAG_PHONE", "OCCUPATION_TYPE", "SELLERPLACE_AREA", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT", "CODE_REJECT_REASON_HC", "AMT_PAYMENT_TOTAL_CURRENT",
    "CHANNEL_TYPE_AP+ (Cash loan)", "NUM_INSTALMENT_NUMBER_x", "CNT_INSTALMENT_MATURE_CUM",
    "CREDIT_TYPE_Car loan", "WEEKDAY_APPR_PROCESS_START_MONDAY",
    "CHANNEL_TYPE_Credit and cash offices", "NAME_YIELD_GROUP_middle",
    "AMT_DRAWINGS_CURRENT", "SK_ID_PREV_y", "CHANNEL_TYPE_Channel of corporate sales",
    "SK_DPD_y", "WEEKDAY_APPR_PROCESS_START_WEDNESDAY", "MONTHS_MAX",
    "NAME_GOODS_CATEGORY_Clothing and Accessories", "CODE_REJECT_REASON_SCO",
    "OBS_30_CNT_SOCIAL_CIRCLE", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "PRODUCT_COMBINATION_Cash Street: middle",
    "NAME_GOODS_CATEGORY_Photo / Cinema Equipment",
    "PRODUCT_COMBINATION_Cash X-Sell: middle", "AMT_APPLICATION", "DAYS_FIRST_DUE",
    "FLAG_DOCUMENT_16", "AMT_ANNUITY", "NAME_SELLER_INDUSTRY_Connectivity",
    "PRODUCT_COMBINATION_POS industry without interest", "NONLIVINGAPARTMENTS_AVG",
    "NAME_SELLER_INDUSTRY_Industry", "DAYS_CREDIT_UPDATE", "RATE_DOWN_PAYMENT",
    "ELEVATORS_AVG", "STATUS_X", "ENTRANCES_AVG",
    "WEEKDAY_APPR_PROCESS_START_SATURDAY", "MONTHS_BALANCE_y",
    "AMT_CREDIT_SUM_OVERDUE", "NAME_PORTFOLIO_POS", "SK_DPD_x", "FLAG_DOCUMENT_13",
    "NAME_TYPE_SUITE_Unaccompanied", "WEEKDAY_APPR_PROCESS_START_FRIDAY",
    "FLAG_DOCUMENT_6", "CHANNEL_TYPE_Stone", "SK_DPD_DEF_y",
    "NAME_PRODUCT_TYPE_x-sell", "NONLIVINGAREA_AVG", "FLAG_DOCUMENT_18",
    "nb_app", "NAME_HOUSING_TYPE", "NAME_PORTFOLIO_Cash",
    "WEEKDAY_APPR_PROCESS_START_SUNDAY", "WEEKDAY_APPR_PROCESS_START_THURSDAY",
    "NFLAG_INSURED_ON_APPROVAL", "MONTHS_COUNT", "YEARS_BUILD_MEDI",
    "DAYS_FIRST_DRAWING", "CODE_REJECT_REASON_LIMIT", "NAME_YIELD_GROUP_XNA",
    "NAME_GOODS_CATEGORY_Medical Supplies", "CREDIT_TYPE_Consumer credit",
    "PRODUCT_COMBINATION_POS household with interest",
    "PRODUCT_COMBINATION_POS mobile with interest", "AMT_DRAWINGS_POS_CURRENT",
    "NAME_CONTRACT_TYPE_Revolving loans", "FLOORSMAX_AVG", "WALLSMATERIAL_MODE",
    "CHANNEL_TYPE_Contact center", "CREDIT_TYPE_Credit card",
    "CODE_REJECT_REASON_SCOFR", "NAME_SELLER_INDUSTRY_XNA", "BASEMENTAREA_AVG",
    "LANDAREA_MEDI", "NAME_SELLER_INDUSTRY_Consumer electronics", "NAME_TYPE_SUITE",
    "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_CHILDREN",
    "PRODUCT_COMBINATION_Cash Street: high", "NAME_GOODS_CATEGORY_Sport and Leisure",
    "NAME_TYPE_SUITE_Children", "NAME_SELLER_INDUSTRY_Construction",
    "NAME_GOODS_CATEGORY_Tourism", "NAME_CASH_LOAN_PURPOSE_Repairs",
    "FLAG_CONT_MOBILE", "WEEKDAY_APPR_PROCESS_START_TUESDAY",
    "AMT_REQ_CREDIT_BUREAU_DAY", "NAME_GOODS_CATEGORY_Mobile",
    "NAME_TYPE_SUITE_Spouse, partner", "REG_REGION_NOT_LIVE_REGION",
    "CREDIT_TYPE_Another type of loan", "COMMONAREA_MEDI", "FONDKAPREMONT_MODE",
    "NAME_GOODS_CATEGORY_Construction Materials", "NAME_PORTFOLIO_XNA",
    "NAME_TYPE_SUITE_Family", "NAME_PAYMENT_TYPE_Non-cash from your account",
    "NAME_CASH_LOAN_PURPOSE_Other", "CNT_FAM_MEMBERS", "CREDIT_ACTIVE_Sold",
    "PRODUCT_COMBINATION_Card Street", "PRODUCT_COMBINATION_Cash",
    "NAME_TYPE_SUITE_Other_B", "NAME_CONTRACT_TYPE_Cash loans", "FLOORSMIN_AVG",
    "CHANNEL_TYPE_Country-wide", "NUM_INSTALMENT_VERSION",
    "NAME_CONTRACT_STATUS_Canceled", "AMT_REQ_CREDIT_BUREAU_MON",
    "NAME_TYPE_SUITE_Other_A", "NAME_SELLER_INDUSTRY_MLM partners",
    "NAME_GOODS_CATEGORY_Gardening", "REG_CITY_NOT_WORK_CITY",
    "CNT_CREDIT_PROLONG", "NUM_INSTALMENT_NUMBER",
    "NAME_GOODS_CATEGORY_Auto Accessories", "STATUS_C", "NAME_GOODS_CATEGORY_Jewelry",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "CHANNEL_TYPE_Car dealer", "HOUSETYPE_MODE",
    "FLAG_MOBIL", "FLAG_EMAIL", "FLAG_EMP_PHONE",
    "LIVE_REGION_NOT_WORK_REGION", "LIVE_CITY_NOT_WORK_CITY",
    "NAME_GOODS_CATEGORY_Fitness", "NAME_GOODS_CATEGORY_Education",
    "NAME_GOODS_CATEGORY_Direct Sales", "NAME_GOODS_CATEGORY_Consumer Electronics",
    "NAME_GOODS_CATEGORY_Computers", "NAME_GOODS_CATEGORY_Audio/Video",
    "NAME_GOODS_CATEGORY_Additional Service", "NAME_GOODS_CATEGORY_Animals",
    "NAME_TYPE_SUITE_Group of people", "NAME_CLIENT_TYPE_Refreshed",
    "NAME_CLIENT_TYPE_XNA", "AMT_DRAWINGS_OTHER_CURRENT",
    "CREDIT_TYPE_Unknown type of loan", "CODE_REJECT_REASON_SYSTEM",
    "CODE_REJECT_REASON_XNA", "CODE_REJECT_REASON_VERIF",
    "CREDIT_TYPE_Real estate loan", "NUNIQUE_STATUS_x", "NUNIQUE_STATUS_y",
    "NUNIQUE_STATUS2_y", "CNT_DRAWINGS_OTHER_CURRENT",
    "REG_REGION_NOT_WORK_REGION", "NAME_GOODS_CATEGORY_House Construction",
    "NAME_GOODS_CATEGORY_Homewares", "NAME_CASH_LOAN_PURPOSE_Buying a new car",
    "NAME_CASH_LOAN_PURPOSE_Buying a used car", "NAME_CASH_LOAN_PURPOSE_Car repairs",
    "NAME_CASH_LOAN_PURPOSE_Education",
    "NAME_CASH_LOAN_PURPOSE_Business development",
    "NAME_CASH_LOAN_PURPOSE_Buying a garage", "FLAG_LAST_APPL_PER_CONTRACT_N",
    "NAME_CASH_LOAN_PURPOSE_Building a house or an annex",
    "NFLAG_LAST_APPL_IN_DAY", "CHANNEL_TYPE_Regional / Local",
    "NAME_SELLER_INDUSTRY_Auto technology", "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "NAME_GOODS_CATEGORY_Other",
    "NAME_GOODS_CATEGORY_Office Appliances", "NAME_GOODS_CATEGORY_Insurance",
    "NAME_GOODS_CATEGORY_Medicine", "NAME_GOODS_CATEGORY_Weapon",
    "NAME_GOODS_CATEGORY_Vehicles", "NAME_CASH_LOAN_PURPOSE_Buying a holiday home / land",
    "NAME_CASH_LOAN_PURPOSE_Buying a home", "NAME_CASH_LOAN_PURPOSE_Hobby",
    "NAME_CASH_LOAN_PURPOSE_Gasification / water supply",
    "NAME_CASH_LOAN_PURPOSE_Furniture", "NAME_CASH_LOAN_PURPOSE_Everyday expenses",
    "NAME_CASH_LOAN_PURPOSE_Money for a third person",
    "NAME_CASH_LOAN_PURPOSE_Payments on other loans",
    "NAME_CASH_LOAN_PURPOSE_Medicine", "NAME_CASH_LOAN_PURPOSE_Journey",
    "NAME_CASH_LOAN_PURPOSE_Refusal to name the goal",
    "NAME_CASH_LOAN_PURPOSE_Purchase of electronic equipment",
    "NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday",
    "NAME_CASH_LOAN_PURPOSE_Urgent needs",
    "NAME_CONTRACT_STATUS_Unused offer",
    "NAME_PAYMENT_TYPE_Cashless from the account of the employer",
    "NAME_CONTRACT_TYPE_XNA",
    "PRODUCT_COMBINATION_POS household without interest",
    "FLAG_DOCUMENT_5", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_2",
    "EMERGENCYSTATE_MODE",
    "PRODUCT_COMBINATION_POS others without interest",
    "PRODUCT_COMBINATION_POS other with interest",
    "PRODUCT_COMBINATION_POS mobile without interest", "CREDIT_DAY_OVERDUE",
    "NAME_SELLER_INDUSTRY_Tourism", "NAME_SELLER_INDUSTRY_Jewelry",
    "PRODUCT_COMBINATION_Card X-Sell", "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_8", "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10", "CREDIT_CURRENCY_currency 4",
    "CREDIT_CURRENCY_currency 3", "CREDIT_CURRENCY_currency 2",
    "CREDIT_ACTIVE_Bad debt", "FLAG_DOCUMENT_11", "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_14", "CREDIT_TYPE_Cash loan (non-earmarked)",
    "CREDIT_TYPE_Loan for the purchase of equipment",
    "CREDIT_TYPE_Loan for purchase of shares (margin lending)",
    "CREDIT_TYPE_Loan for business development",
    "CREDIT_TYPE_Interbank credit",
    "CREDIT_TYPE_Loan for working capital replenishment",
    "CREDIT_TYPE_Mobile operator loan", "FLAG_OWN_REALTY", "FLAG_OWN_CAR",
]
df = df.drop(columns=DROP_COLUMNS_HC)

NUM_COLS_HC = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT_x", "AMT_ANNUITY_x", "AMT_GOODS_PRICE_x",
    "REGION_POPULATION_RELATIVE", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "OWN_CAR_AGE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START_x", "REG_CITY_NOT_LIVE_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "APARTMENTS_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG",
    "COMMONAREA_AVG", "LANDAREA_AVG", "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG",
    "APARTMENTS_MODE", "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", "ENTRANCES_MODE",
    "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE",
    "LIVINGAREA_MODE", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE",
    "APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI",
    "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAREA_MEDI", "TOTALAREA_MODE", "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_LAST_PHONE_CHANGE", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_ANNUITY_y",
    "AMT_CREDIT_y", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE_y",
    "HOUR_APPR_PROCESS_START_y", "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION", "CNT_PAYMENT",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
    "NAME_CONTRACT_TYPE_Consumer loans", "NAME_CASH_LOAN_PURPOSE_XAP",
    "NAME_CASH_LOAN_PURPOSE_XNA", "NAME_CONTRACT_STATUS_Approved",
    "NAME_CONTRACT_STATUS_Refused", "NAME_PAYMENT_TYPE_Cash through the bank",
    "NAME_PAYMENT_TYPE_XNA", "CODE_REJECT_REASON_CLIENT",
    "CODE_REJECT_REASON_XAP", "NAME_CLIENT_TYPE_New",
    "NAME_CLIENT_TYPE_Repeater", "NAME_GOODS_CATEGORY_Furniture",
    "NAME_GOODS_CATEGORY_XNA", "NAME_PORTFOLIO_Cards", "NAME_PORTFOLIO_Cars",
    "NAME_PRODUCT_TYPE_XNA", "NAME_PRODUCT_TYPE_walk-in",
    "NAME_SELLER_INDUSTRY_Clothing", "NAME_SELLER_INDUSTRY_Furniture",
    "NAME_YIELD_GROUP_high", "NAME_YIELD_GROUP_low_action",
    "NAME_YIELD_GROUP_low_normal", "PRODUCT_COMBINATION_Cash Street: low",
    "PRODUCT_COMBINATION_Cash X-Sell: high", "PRODUCT_COMBINATION_Cash X-Sell: low",
    "PRODUCT_COMBINATION_POS industry with interest", "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "AMT_CREDIT_MAX_OVERDUE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT",
    "STATUS_0", "STATUS_1", "STATUS_2", "STATUS_3", "STATUS_4", "STATUS_5",
    "MONTHS_MIN", "CREDIT_ACTIVE_Active", "CREDIT_ACTIVE_Closed",
    "CREDIT_CURRENCY_currency 1", "CREDIT_TYPE_Microloan", "CREDIT_TYPE_Mortgage",
    "buro_count", "MONTHS_BALANCE_x", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_DEF_x", "NUNIQUE_STATUS2_x", "AMT_BALANCE",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT", "NUM_INSTALMENT_VERSION_x",
    "DAYS_INSTALMENT_x", "DAYS_ENTRY_PAYMENT_x", "AMT_INSTALMENT_x",
    "AMT_PAYMENT_x", "SK_ID_PREV_x", "NUM_INSTALMENT_VERSION_y",
    "NUM_INSTALMENT_NUMBER_y", "DAYS_INSTALMENT_y", "DAYS_ENTRY_PAYMENT_y",
    "AMT_INSTALMENT_y", "AMT_PAYMENT_y", "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
]
CAT_COLS_HC = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE", "EMERGENCYSTATE_MODE",
]
num_cols = NUM_COLS_HC
cat_cols = CAT_COLS_HC

## 3. Preprocessing Pipeline

In [ ]:
def apply_missing_value_filter(X_train, X_valid, X_test, threshold=0.50):
    """Drop columns whose missing rate in X_train exceeds `threshold`."""
    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
    print("--- 2. Missing Value Column Filter ---")
    print(f"Missing percentage threshold: {threshold:.1%}")
    print(f"Columns dropped ({len(cols_to_drop)}): {cols_to_drop if cols_to_drop else 'None'}")

    X_train_clean = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=cols_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=cols_to_drop, errors="ignore")

    all_cols = X_train_clean.columns.tolist()
    num_cols_clean = X_train_clean.select_dtypes(include=np.number).columns.tolist()
    cat_cols_clean = [c for c in all_cols if c not in num_cols_clean]
    print(f"Features remaining: {X_train_clean.shape[1]}")
    print("-" * 50)
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def apply_correlation_drop(X_train, y_train, X_valid, X_test, threshold=0.9):
    """Drop one feature from each numeric pair whose |correlation| > threshold,
    keeping the one with higher absolute correlation to the target."""
    print("--- 3. Correlation Drop (numeric only) ---")
    all_cols = X_train.columns.tolist()
    num_cols_start = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols_start = [c for c in all_cols if c not in num_cols_start]
    X_train_num = X_train[num_cols_start]

    print(f"Numeric features before drop: {len(num_cols_start)}. Threshold: {threshold}")
    corr_matrix = X_train_num.corr().abs()
    target_corr = X_train_num.apply(lambda x: x.corr(y_train)).abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop_num = set()
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                a, b = row, col
                if a in to_drop_num or b in to_drop_num:
                    continue
                if target_corr.get(a, 0) < target_corr.get(b, 0):
                    to_drop_num.add(a)
                else:
                    to_drop_num.add(b)

    features_to_drop = list(to_drop_num)
    X_train_clean = X_train.drop(columns=features_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=features_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=features_to_drop, errors="ignore")
    print(f"Numeric features dropped: {len(features_to_drop)}")
    print("-" * 50)

    num_cols_clean = [c for c in num_cols_start if c not in features_to_drop]
    cat_cols_clean = cat_cols_start
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.50):
    """Full HC preprocessing pipeline: split, filter missing, drop correlated,
    impute, label-encode, scale. Returns processed splits and helper objects."""
    print("--- 1. Splitting Data (64/16/20) ---")
    X = df.drop(columns=[target])
    y = df[target]
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42,
    )
    print(f"Split sizes: Train={len(X_train)} | Valid={len(X_valid)} | Test={len(X_test)}")
    print("-" * 50)

    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    X_train, X_valid, X_test, num_cols, cat_cols = apply_missing_value_filter(
        X_train, X_valid, X_test, threshold=missing_threshold,
    )
    X_train, X_valid, X_test, num_cols, cat_cols = apply_correlation_drop(
        X_train, y_train, X_valid, X_test, threshold=corr_threshold,
    )

    print("--- 4. Leakage-Free Transformations ---")
    median_imputers = {}
    if num_cols:
        print(f"Imputing {len(num_cols)} numeric columns with median...")
        for col in num_cols:
            median_value = X_train[col].median()
            median_imputers[col] = median_value
            X_train[col] = X_train[col].fillna(median_value)
            X_valid[col] = X_valid[col].fillna(median_value)
            X_test[col]  = X_test[col].fillna(median_value)

    encoders = {}
    cat_cardinalities = []
    for col in cat_cols:
        X_train[col] = X_train[col].fillna("Missing")
        X_valid[col] = X_valid[col].fillna("Missing")
        X_test[col]  = X_test[col].fillna("Missing")

        le = LabelEncoder()
        le.fit(X_train[col])
        new_classes = list(le.classes_)
        if "Missing" not in new_classes:
            new_classes.append("Missing")
        new_classes.append("UNKNOWN")
        le.classes_ = np.array(new_classes)
        X_train[col] = le.transform(X_train[col])
        encoders[col] = le
        cat_cardinalities.append(len(le.classes_))

        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        X_valid[col] = (X_valid[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))
        X_test[col]  = (X_test[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))

    print(f"Categorical features encoded: {len(cat_cols)}")

    scaler = StandardScaler()
    if num_cols:
        print(f"Scaling {len(num_cols)} numeric columns...")
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_valid[num_cols] = scaler.transform(X_valid[num_cols])
        X_test[num_cols]  = scaler.transform(X_test[num_cols])
    print("-" * 50)

    print("--- Final Feature Summary ---")
    print(f"Numeric features      : {len(num_cols)}")
    print(f"Categorical features  : {len(cat_cols)}")
    print(f"Cat cardinalities     : {cat_cardinalities}")
    print(f"Total features        : {X_train.shape[1]}")
    print("-" * 50)

    return (
        X_train, y_train, X_valid, y_valid, X_test, y_test,
        num_cols, cat_cols, encoders, median_imputers, scaler, cat_cardinalities,
    )


def home_credit_summary(df, num_cols, cat_cols, target="TARGET"):
    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total Rows: {df.shape[0]:,}")
    print(f"Total Columns: {df.shape[1]:,}\n")
    print("Feature Breakdown")
    print(f"  Numeric Features     : {len(num_cols)}")
    print(f"  Categorical Features : {len(cat_cols)}")
    print(f"  Binary FLAG Columns  : {len([c for c in df.columns if 'FLAG' in c])}")
    print("-" * 50)
    if target in df.columns:
        target_dist = df[target].value_counts(normalize=True) * 100
        print("Target Distribution")
        print(target_dist.to_frame("Percentage %"))
        print("-" * 50)
    print("Missing Value Summary (Top 15)")
    missing = df.isnull().mean().sort_values(ascending=False)
    print((missing * 100).head(15).to_frame("Missing %"))
    print("=" * 60)


home_credit_summary(df, num_cols, cat_cols)
(
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    final_num_cols, final_cat_cols, encoders, imputers, scaler, cat_cardinalities_final,
) = preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.9)

print("\n================ Dataset Split Summary ================\n")
print(f"Train : X = {X_train.shape},  y = {y_train.shape}")
print(f"Valid : X = {X_valid.shape},  y = {y_valid.shape}")
print(f"Test  : X = {X_test.shape},   y = {y_test.shape}")
print(f"\nNumeric features      : {len(final_num_cols)}")
print(f"Categorical features  : {len(final_cat_cols)}")
print(f"Cat cardinalities     : {cat_cardinalities_final}")

## 4. PCA Whitening on Numerical Features

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(whiten=True, n_components=0.99)

val_num = X_valid[final_num_cols].copy()
test_num = X_test[final_num_cols].copy()
train_num = X_train[final_num_cols].copy()
print(f' num of numerical features before pca {len(train_num.columns)}')
pca_features = pca.fit_transform(train_num)
pc_columns = [f'PC{i+1}' for i in range(pca_features.shape[1])]
print(f' num of numerical features after pca {len(pc_columns)}')
num_df = pd.DataFrame(
    pca_features,
    columns=pc_columns,
    index=train_num.index
)
original_index = val_num.index
features_transformed = pca.transform(val_num)
val_df_num = pd.DataFrame(
                          features_transformed,
                          columns=pc_columns,
                          index=original_index)
original_index = test_num.index
features_transformed = pca.transform(test_num)
test_df_num = pd.DataFrame(
                          features_transformed,
                          columns=pc_columns,
                          index=original_index)

## 5. Scree Plot — Cumulative Explained Variance

In [ ]:
import matplotlib.pyplot as plt

pca = PCA()
pca_num_df = X_train[final_num_cols].copy()
pca_num_df[pca_num_df.columns] = scaler.fit_transform(pca_num_df[pca_num_df.columns])
pca.fit(pca_num_df)
d_numerical = len(final_num_cols)
exp_var_pca = pca.explained_variance_ratio_

# Cumulative sum: how many components do we need to reach 95% variance?
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

plt.figure(figsize=(10, 5))

# Individual Variance
plt.bar(range(1, d_numerical + 1), exp_var_pca, alpha=0.5, align='center', label='Individual Explained Variance', color='skyblue')

# Cumulative Variance
plt.step(range(1, d_numerical + 1), cum_sum_eigenvalues, where='mid', label='Cumulative Explained Variance', color='red', linewidth=1)

plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Component Index')
plt.title('Scree Plot: Explained Variance Ratio (Simulated Lending Data)')
plt.xticks(range(1, d_numerical + 1))
plt.legend(loc='best')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(str(FIGURES_DIR / 'pca_scree_plot_hc.png'), dpi=150, bbox_inches='tight')
plt.show()
# plt.savefig('scree_plot.png')
# plt.close()

# Print values for interpretation
print(f"Top 70 components explain: {np.sum(exp_var_pca[:70]):.2%}")
print(f"Top 84 components explain: {np.sum(exp_var_pca[:84]):.2%}")
del pca_num_df

## 6. Build XGB Frames

We build two frames — one with the original (non-PCA) numerical columns and one with the PCA-whitened ones — both with the same categorical encoding (category dtype, label-encoded categories) so XGBoost can use `enable_categorical=True`.

In [ ]:
train_df_xgb_pca = pd.concat([num_df.reset_index(drop=True), X_train[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)
val_df_xgb_pca = pd.concat([val_df_num.reset_index(drop=True), X_valid[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)
test_df_xgb_pca = pd.concat([test_df_num.reset_index(drop=True), X_test[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)

In [ ]:
for col in final_cat_cols:
    all_cats = pd.concat([
        train_df_xgb_pca[col],
        val_df_xgb_pca[col]
    ]).astype("category").cat.categories
    train_df_xgb_pca[col] = pd.Categorical(
        train_df_xgb_pca[col],
        categories=all_cats
    )
    val_df_xgb_pca[col] = pd.Categorical(
        val_df_xgb_pca[col],
        categories=all_cats
    )
    test_df_xgb_pca[col] = pd.Categorical(
        test_df_xgb_pca[col],
        categories=all_cats
    )

In [ ]:
train_df_xgb = pd.concat([ X_train[final_num_cols].reset_index(drop=True), X_train[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)
val_df_xgb = pd.concat([ X_valid[final_num_cols].reset_index(drop=True), X_valid[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)
test_df_xgb = pd.concat([ X_test[final_num_cols].reset_index(drop=True), X_test[final_cat_cols].reset_index(drop=True).astype('category')], axis=1)

In [ ]:
for col in final_cat_cols:
    all_cats = pd.concat([
        train_df_xgb[col],
        val_df_xgb[col]
    ]).astype("category").cat.categories
    train_df_xgb[col] = pd.Categorical(
        train_df_xgb[col],
        categories=all_cats
    )
    val_df_xgb[col] = pd.Categorical(
        val_df_xgb[col],
        categories=all_cats
    )
    test_df_xgb[col] = pd.Categorical(
        test_df_xgb[col],
        categories=all_cats
    )

## 7. XGBoost — Baseline (Raw Features)

In [ ]:
EARLY_STOPPING_ROUNDS = 50
best_params = {'n_estimators': 2987,
 'learning_rate': 0.020396329269675148,
 'max_depth': 2,
 'lambda': 4.108196683393735,
 'alpha': 3.7989498214701167,
 'scale_pos_weight': 5.9196140867431,
 'min_child_weight': 30,
 'gamma': 0.7731526376179212,
 'subsample': 0.9500946446141826,
 'colsample_bytree': 0.949320170113834}
final_xgb_config = {
    **best_params, # Unpack the parameters found by Optuna
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "enable_categorical" : True,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    'device': 'cuda'
}
final_eval_set = [
    (train_df_xgb, y_train),
    (val_df_xgb, y_valid) # Validation set determines the stopping point
]
eval_names = ['train', 'validation']

In [ ]:
import xgboost as xgb
model_xgb = xgb.XGBClassifier(**final_xgb_config)

model_xgb.fit(train_df_xgb, y_train,
    eval_set=final_eval_set,
    verbose=200)

val_pred = model_xgb.predict_proba(val_df_xgb)[:, 1]
roc_auc_score(y_valid, val_pred)

## 8. XGBoost — PCA-Whitened Features

In [ ]:
import xgboost as xgb
model_xgb_pca = xgb.XGBClassifier(**final_xgb_config)

model_xgb_pca.fit(
    train_df_xgb_pca, y_train,
    eval_set=[(val_df_xgb_pca, y_valid)],
    verbose=200,
)

val_pred = model_xgb_pca.predict_proba(val_df_xgb_pca)[:, 1]
roc_auc_score(y_valid, val_pred)

## 9. FT-Transformer Setup with PCA Features

In [ ]:
train_df_proc = pd.concat([num_df.reset_index(drop=True), X_train[final_cat_cols].reset_index(drop=True),y_train.reset_index(drop=True)], axis=1)
print('encoded train')

In [ ]:
final_num_cols_pca = list(val_df_num.columns)
val_df = pd.concat([val_df_num.reset_index(drop=True), X_valid[final_cat_cols].reset_index(drop=True), y_valid.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df_num.reset_index(drop=True), X_test[final_cat_cols].reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)
# val_df = val_df_pca_ftt[final_num_cols_pca+final_cat_cols+['TARGET']]
# print(final_num_cols_pca)
d_numerical = len(final_num_cols_pca)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_numpy = { "train": {},'val':{},'test':{}}
data_numpy["train"]["x_cat"] = train_df_proc[final_cat_cols]
data_numpy["val"]["x_cat"] = val_df[final_cat_cols]
data_numpy["test"]["x_cat"] = test_df[final_cat_cols]

data_numpy["train"]["X_cont"] = train_df_proc[final_num_cols_pca]
data_numpy["val"]["X_cont"] = val_df[final_num_cols_pca]
data_numpy["test"]["X_cont"] = test_df[final_num_cols_pca]

data_numpy["train"]["y"] = train_df_proc['TARGET']
data_numpy["val"]["y"] = val_df['TARGET']
data_numpy["test"]["y"] = test_df['TARGET']

# data = {
# part: {k: torch.as_tensor(v.to_numpy()).to(device) for k, v in data_numpy[part].items()}
# for part in data_numpy
# }
d_numerical = len(final_num_cols_pca)

In [ ]:
data={}
for part in data_numpy:
    data[part] = {}

    # numerical features
    data[part]["X_cont"] = torch.tensor(
        data_numpy[part]["X_cont"].to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    # categorical features
    data[part]["x_cat"] = torch.tensor(
        data_numpy[part]["x_cat"].to_numpy(),
        dtype=torch.long,
        device=device,
    )

    # labels
    data[part]["y"] = torch.tensor(
        data_numpy[part]["y"].to_numpy(),
        dtype=torch.float32,
        device=device,
    )

## 10. FTT Training

In [ ]:
BATCH_SIZE=2048+4096
trial_results=[]
global_best_auc=0.774
import math
from rtdl_revisiting_models import FTTransformer

global_model_path = str(MODELS_DIR / "ftt_pca_hc.pth")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=3, #4
    d_block=64, #192
    attention_n_heads=4,#8
    ffn_d_hidden_multiplier=2,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)
n_epochs = 2 if DEV_MODE else 100
patience = 10
Lr = 0.0002
optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)

loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return auc # The higher -- the better.


print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


batch_size = BATCH_SIZE
epoch_size = math.ceil(train_df_proc.shape[0] / batch_size)
timer = delu.tools.Timer()
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            torch.tensor(6.0, device=y_batch.device),
            torch.tensor(1.0, device=y_batch.device)
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    # test_auc = evaluate("test")
    train_auc = evaluate('train')
    trial_results.append({
    "epoch_n": epoch,
    "auc_val": val_auc,
    # "auc_test": test_auc,
    'best':best,
    'time':timer.__str__(),
    'auc_train':train_auc
      })
    # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} [time] {timer}")

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "epoch": epoch, 'train':train_auc}
        if val_auc>global_best_auc:
          torch.save(model.state_dict(), global_model_path)
          global_best_auc=val_auc

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    print()

In [ ]:
torch.save(model.state_dict(), str(MODELS_DIR / "ftt_pca_hc.pth"))

## 11. Evaluation Helpers

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

@torch.no_grad()
def evaluate_metrics(model, dataloader, device="cuda"):
    model.eval()
    all_probs = []
    all_labels = []

    for batch in dataloader:
        X_num = batch["X_cont"].to(device)
        X_cat = batch["x_cat"].to(device)
        y = batch["y"].cpu().numpy()

        logits = model(X_num, X_cat)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(y)
    print('computed y_prob')

    y_true = np.concatenate(all_labels)
    y_prob = np.concatenate(all_probs).reshape(-1)

    # ---- Metrics ----
    # AUC
    auc = roc_auc_score(y_true, y_prob)
    print('computed auc')

    # KS statistic (max separation between TPR and FPR)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks = np.max(tpr - fpr)
    print('computed ks')

    # Gini
    gini = 2 * auc - 1
    print('computed gini')

    # Precision & Recall @10% capture rate
    threshold_10pct = np.percentile(y_prob, 90)

    # 2. Generate binary predictions based on that top 10% volume
    y_pred_at_10 = (y_prob >= threshold_10pct).astype(int)

    # 3. Calculate Precision and Recall for this specific slice
    # In Credit Risk, Recall @ 10% is also known as the "Capture Rate"
    precision_at_10 = precision_score(y_true, y_pred_at_10)
    recall_at_10 = recall_score(y_true, y_pred_at_10)

    # 4. Standard PR Curve (using full probabilities for the curve)
    # Note: Pass y_prob, not y_pred, to get the full curve!
    # precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, y_prob)

    auc_roc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob) # Also known as Area Under PR Curve


    df = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    df = df.sort_values('y_prob', ascending=False).reset_index(drop=True)
    df['decile'] = pd.qcut(df.index, 10, labels=False) # Divide into 10 equal sized bins

    overall_ratio = sum(y_true) / len(y_true)
    lift_data = []
    for i in range(10):
        decile_df = df[df['decile'] == i]
        min_p = decile_df['y_prob'].min()
        max_p = decile_df['y_prob'].max()
        total_apps = len(decile_df)
        bad_count = int(decile_df['y_true'].sum())
        good_count = total_apps - bad_count
        decile_ratio = decile_df['y_true'].mean()
        lift = decile_ratio / overall_ratio
        lift_data.append({"bin": i + 1,
                          "min_prob": round(min_p, 4),
                          "max_prob": round(max_p, 4),
                          "Total_Apps": total_apps,
                          "Bad_Count": bad_count,
                          "Good_Count": good_count,
                          "lift": lift,
                          "bad_rate": decile_ratio})
        # # Precision & Recall at default 0.5 threshold
        # y_pred = (y_prob >= 0.15).astype(int)
        # precision = precision_score(y_true, y_pred)
        # recall = recall_score(y_true, y_pred)
        # precision_curve, recall_curve, thresholds = precision_recall_curve(y_true, y_pred)
        # # auprc = average_precision_score(y_true, y_prob)

        # # ---- Threshold-free single score ----
        # auprc = average_precision_score(y_true, y_prob)

        # # AUPRC (area under Precision-Recall curve)
        # auprc = average_precision_score(y_true, y_prob)

    return {
        "AUC": auc,
        "KS": ks,
        "GINI": gini,
        "Precision": precision_at_10,
        "Recall": recall_at_10,
        # "precision_curve": precision_curve,
        # "recall_curve": recall_curve,
        # "thresholds": thresholds,
        "AUPRC": auprc
    }, pd.DataFrame(lift_data)

In [ ]:
import torch

class TabDataset(torch.utils.data.Dataset):
    def __init__(self, X_cont, x_cat, y):
        self.X_cont = (
            X_cont.float()
            if torch.is_tensor(X_cont)
            else torch.tensor(X_cont, dtype=torch.float32)
        )

        self.x_cat = (
            x_cat.long()
            if torch.is_tensor(x_cat)
            else torch.tensor(x_cat, dtype=torch.long)
        )

        self.y = (
            y.float()
            if torch.is_tensor(y)
            else torch.tensor(y, dtype=torch.float32)
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "X_cont": self.X_cont[idx],
            "x_cat": self.x_cat[idx],
            "y": self.y[idx],
        }

from torch.utils.data import DataLoader

train_ds = TabDataset(**data["train"])
val_ds = TabDataset(**data["val"])
test_ds = TabDataset(**data["test"])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
model = FTTransformer(
  n_cont_features=d_numerical,
  cat_cardinalities=cat_cardinalities_final,
  n_blocks=3,
  d_block=64,
  attention_n_heads=4,
  ffn_d_hidden_multiplier=2,#
  attention_dropout=0.05,#Attention_dropout
  ffn_dropout=0.05,#FFN_dropout
  residual_dropout=0.05,#Residual_droupout
  d_out=1 # output a single logit (binary)
).to(device)
state = torch.load("ftt_rot_home_64_B3_H4_M2.pth", map_location=torch.device('cpu'))
model.load_state_dict(state)
model.to(device)
# print('sent to device')
# Evaluate
metrics, lift_val = evaluate_metrics(model, val_loader, device)
perf_val = pd.DataFrame(metrics, index=['Value'])# Transpose for a cleaner, vertical display
print("Evaluation Metrics:")
print(perf_val)
lift_val

## 12. Test & Train Metrics

In [ ]:
metrics, lift_test = evaluate_metrics(model, test_loader, device)
print("Evaluation Metrics:")
# for k, v in metrics.items():
# print(f"{k}: {v:.5f}")
perf_test = pd.DataFrame(metrics, index=['Value'])# Transpose for a cleaner, vertical display
print(perf_test)
lift_test

In [ ]:
metrics, lift_train = evaluate_metrics(model, train_loader, device)
print("Evaluation Metrics:")
# for k, v in metrics.items():
# print(f"{k}: {v:.5f}")
perf_train = pd.DataFrame(metrics, index=['Value'])# Transpose for a cleaner, vertical display
print(perf_train)
lift_train

## 13. Probability Blending (FTT + XGB)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import xgboost as xgb
import torch
import scipy.special

# ── Step 1: Get OOF predictions from both models ──
# These should be generated during training on train set
# Never use test set for finding optimal weights

def get_ftt_oof_predictions(model, data, batch_size, device):
    """Get FTT predictions on a dataset split."""
    model.eval()
    with torch.no_grad():
        y_pred = torch.cat([
            apply_model(batch)
            for batch in delu.iter_batches(data, batch_size)
        ]).cpu().numpy()
    return scipy.special.expit(y_pred) # convert logits to probs

def get_xgb_oof_predictions(xgb_model, X):
    """Get XGB predictions."""
    # val_pred = model_xgb.predict_proba(val_df_xgb.drop(columns=['target_binary']))[:, 1]
    # test_pred = model_xgb.predict_proba(test_df_xgb.drop(columns=['target_binary']))[:, 1]
    # dmatrix = xgb.DMatrix(X)
    # return xgb_model.predict(dmatrix)
    return xgb_model.predict_proba(X)[:, 1]


# ── Step 2: Load best saved FTT model ──
# base_model = FTTransformer(
# n_cont_features=d_numerical,
# cat_cardinalities=cat_cardinalities_final,
# n_blocks=3,
# d_block=64,
# attention_n_heads=4,
# ffn_d_hidden_multiplier=2,
# attention_dropout=0.05,
# ffn_dropout=0.05,
# residual_dropout=0.05,
# d_out=1
# )
# ftt_model = torch.nn.DataParallel(base_model, device_ids=[0,1]).to(device)
# ftt_model.load_state_dict(torch.load(global_model_path))
# ftt_model.eval()


# ── Step 3: Generate predictions on val and test ──
ftt_val_preds = get_ftt_oof_predictions(model, data["val"], BATCH_SIZE, device)
ftt_test_preds = get_ftt_oof_predictions(model, data["test"], BATCH_SIZE, device)

xgb_val_preds = get_xgb_oof_predictions(model_xgb_pca, val_df_xgb_pca)
xgb_test_preds = get_xgb_oof_predictions(model_xgb_pca, test_df_xgb_pca)

y_val = data["val"]["y"].cpu().numpy()
y_test = data["test"]["y"].cpu().numpy()

print(f"FTT Val AUC: {roc_auc_score(y_val, ftt_val_preds):.4f}")
print(f"XGB Val AUC: {roc_auc_score(y_val, xgb_val_preds):.4f}")


# ── Step 4: Find optimal ensemble weights on val set ──
def ensemble_auc(weights, preds_list, y_true):
    """Compute AUC for weighted ensemble."""
    weights = np.array(weights)
    weights = weights / weights.sum() # normalize to sum to 1
    blended = sum(w * p for w, p in zip(weights, preds_list))
    return -roc_auc_score(y_true, blended) # negative because we minimize

# Optimize weights using val set
result = minimize(
    ensemble_auc,
    x0=[0.5, 0.5], # initial equal weights
    args=([ftt_val_preds, xgb_val_preds], y_val),
    method='Nelder-Mead',
    options={'maxiter': 1000, 'xatol': 1e-6}
)

optimal_weights = np.array(result.x)
optimal_weights = optimal_weights / optimal_weights.sum()
ftt_weight, xgb_weight = optimal_weights

print(f"\nOptimal weights — FTT: {ftt_weight:.3f} | XGB: {xgb_weight:.3f}")


# ── Step 5: Evaluate ensemble ──
def blend(preds_list, weights):
    weights = np.array(weights) / np.array(weights).sum()
    return sum(w * p for w, p in zip(weights, preds_list))

# Optimal weighted blend
val_blend = blend([ftt_val_preds, xgb_val_preds], [ftt_weight, xgb_weight])
test_blend = blend([ftt_test_preds, xgb_test_preds], [ftt_weight, xgb_weight])

# Simple average as comparison
val_simple = blend([ftt_val_preds, xgb_val_preds], [0.5, 0.5])
test_simple = blend([ftt_test_preds, xgb_test_preds], [0.5, 0.5])

print(f"\n── Val Results ──")
print(f"Simple average : {roc_auc_score(y_val, val_simple):.4f}")
print(f"Optimal weights : {roc_auc_score(y_val, val_blend):.4f}")

print(f"\n── Test Results ──")
print(f"Simple average : {roc_auc_score(y_test, test_simple):.4f}")
print(f"Optimal weights : {roc_auc_score(y_test, test_blend):.4f}")


# ── Step 6: Rank correlation between models ──
# High correlation = less benefit from ensembling
from scipy.stats import spearmanr
corr, _ = spearmanr(ftt_val_preds, xgb_val_preds)
print(f"\nSpearman rank correlation FTT vs XGB: {corr:.4f}")
print(f"Diversity (1 - corr): {1-corr:.4f}")